In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import json
import numpy as np
from datasets import load_dataset
import tqdm
import matplotlib.pyplot as plt
import random

index = faiss.read_index("my_rag_db.index")
index.nprobe = 16  # number of clusters to visit for IVF search

with open("my_rag_db.json", "r") as f:
    metadata = json.load(f)

# Build set of indexed URLs for filtering
indexed_urls = set(m["url"] for m in metadata)

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Load the same 100k documents used for indexing (non-streaming)
print("Loading dataset...")
dataset = load_dataset("natural_questions", split="train[:100000]")

In [ ]:
NB_QUESTIONS_TEST = 500

# Build candidate pool: only questions whose document URL is in the index
candidates = []
for example in tqdm.tqdm(dataset, desc="Filtering candidates"):
    url = example["document"]["url"]
    if url in indexed_urls:
        candidates.append({
            "question": example["question"]["text"],
            "ground_truth_url": url
        })

print(f"Candidates with URL in index: {len(candidates)} / {len(dataset)}")

# Random sample
random.seed(42)
eval_set = random.sample(candidates, min(NB_QUESTIONS_TEST, len(candidates)))
print(f"Eval set: {len(eval_set)} questions (random sample)")

In [28]:
import numpy as np

all_relevance_results = []
K_TOP = 20

for question in eval_set:
    embedding = embedder.encode([question["question"]], convert_to_numpy=True)
    faiss.normalize_L2(embedding)
    distances, indices = index.search(embedding, K_TOP)
    
    current_query_relevance = []
    
    for idx in indices[0]:
        found_url = metadata[idx]["url"]
        is_relevant = (found_url == question["ground_truth_url"])
        current_query_relevance.append(is_relevant)
    
    all_relevance_results.append(current_query_relevance)

all_relevance_results[0]
    


[True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False]

In [ ]:
def calculate_ap(relevance_bools):
    precisions = []
    num_relevant_found = 0
    
    for i, is_relevant in enumerate(relevance_bools):
        rank = i + 1
        
        if is_relevant:
            num_relevant_found += 1
            precision_at_rank = num_relevant_found / rank
            precisions.append(precision_at_rank)
    
    if not precisions:
        return 0.0
    
    return sum(precisions) / len(precisions)


def calculate_recall_at_k(relevance_bools, k):
    """Returns 1 if at least one relevant document is in the top-k, else 0."""
    return 1.0 if any(relevance_bools[:k]) else 0.0


def calculate_rr(relevance_bools):
    """Reciprocal rank: 1/rank of the first relevant document."""
    for i, is_relevant in enumerate(relevance_bools):
        if is_relevant:
            return 1.0 / (i + 1)
    return 0.0


# --- MAP@20 ---
ap_scores = [calculate_ap(res) for res in all_relevance_results]
map_score = sum(ap_scores) / len(ap_scores)

# --- Recall@K ---
recall_at_1 = np.mean([calculate_recall_at_k(res, 1) for res in all_relevance_results])
recall_at_5 = np.mean([calculate_recall_at_k(res, 5) for res in all_relevance_results])
recall_at_20 = np.mean([calculate_recall_at_k(res, 20) for res in all_relevance_results])

# --- MRR ---
rr_scores = [calculate_rr(res) for res in all_relevance_results]
mrr_score = np.mean(rr_scores)

print(f"Nombre de questions: {len(ap_scores)}")
print(f"MAP@{K_TOP}    : {map_score:.3f}")
print(f"MRR        : {mrr_score:.3f}")
print(f"Recall@1   : {recall_at_1:.3f}")
print(f"Recall@5   : {recall_at_5:.3f}")
print(f"Recall@20  : {recall_at_20:.3f}")

In [ ]:
def interpolated_precision_at_recall(relevance_bools, recall_levels):
    """Compute interpolated precision at 11 standard recall levels for a single query."""
    # Compute precision and recall at each rank
    precisions_at_ranks = []
    recalls_at_ranks = []
    num_relevant_found = 0
    total_relevant = sum(relevance_bools)
    
    if total_relevant == 0:
        return [0.0] * len(recall_levels)
    
    for i, is_relevant in enumerate(relevance_bools):
        rank = i + 1
        if is_relevant:
            num_relevant_found += 1
        precision = num_relevant_found / rank
        recall = num_relevant_found / total_relevant
        precisions_at_ranks.append(precision)
        recalls_at_ranks.append(recall)
    
    # Interpolate: for each recall level r, precision = max(precision at recall >= r)
    interpolated = []
    for r in recall_levels:
        max_prec = 0.0
        for prec, rec in zip(precisions_at_ranks, recalls_at_ranks):
            if rec >= r:
                max_prec = max(max_prec, prec)
        interpolated.append(max_prec)
    
    return interpolated


# 11 standard recall levels
recall_levels = np.linspace(0.0, 1.0, 11)

# Compute interpolated precision for each query, then average
all_interpolated = [interpolated_precision_at_recall(res, recall_levels) for res in all_relevance_results]
avg_interpolated = np.mean(all_interpolated, axis=0)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(recall_levels, avg_interpolated, marker='o', linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Interpolated Precision")
plt.title("11-Point Interpolated Precision-Recall Curve")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.grid(True, alpha=0.3)
plt.xticks(recall_levels)
plt.tight_layout()
plt.show()